# 003 · Executable Examples with doctest

> **Standard Library › Testing & Debugging** · Python 3.14

`doctest` finds snippets that look like interactive Python sessions (`>>> ...`) inside docstrings or text files, runs them, and checks that the output matches what is written. It keeps usage examples in your documentation **honest**: if the code changes and the example no longer works, the doctest fails.

**In this notebook you will learn:**

- how to write examples in docstrings that double as tests
- how to run them in a notebook (`run_docstring_examples`, `DocTestFinder` / `DocTestRunner`) and in scripts (`testmod`, `python -m doctest`)
- the exact output-matching rules, and the `ELLIPSIS` / `NORMALIZE_WHITESPACE` directives
- how to document expected exceptions
- how to test documentation in `.txt` files with `testfile`, and plug doctests into `unittest`
- when doctest is the right tool — and when it is not

## .NET mental model

There is **no real .NET equivalent**. The closest thing is the `<example>` / `<code>` section of an XML doc comment — but that is documentation only; nothing ever compiles or runs it, so it silently goes stale.

```csharp
/// <summary>Doubles a value.</summary>
/// <example>
/// <code>
/// var x = Calc.Double(4);   // 8 — but nobody checks this!
/// </code>
/// </example>
public static int Double(int x) => 2 * x;
```

| Concept | .NET | Python |
|---|---|---|
| Examples in API docs | XML doc `<example>` (not executed) | `>>>` examples in docstrings (**executed by doctest**) |
| Verify docs/tutorials | manual, or a custom docs build step | `doctest.testfile("guide.txt")` |
| Run as part of the test suite | — | `doctest.DocTestSuite(module)` inside `unittest` |

## Anatomy of a doctest

A doctest is a copy of an interactive session pasted into a docstring:

- `>>> ` starts a statement, `... ` continues it (loops, `def`, multi-line calls)
- the lines that follow, up to the next `>>>` or a blank line, are the **expected output**
- an expression's value is shown via its `repr()` — exactly as the REPL would show it; `None` shows nothing

In [ ]:
def apply_discount(price: float, percent: float) -> float:
    """Return *price* reduced by *percent* percent, rounded to cents.

    >>> apply_discount(100, 10)
    90.0
    >>> apply_discount(19.99, 0)
    19.99
    >>> for p in (25, 50):
    ...     print(p, apply_discount(80, p))
    25 60.0
    50 40.0
    """
    if not 0 <= percent <= 100:
        raise ValueError(f"percent must be 0-100, got {percent}")
    return round(price * (1 - percent / 100), 2)

print(apply_discount.__doc__.count(">>>"), "examples in the docstring")

## Running doctests in a notebook

The quickest way to check a single function is `doctest.run_docstring_examples(obj, globs)`. Pass `globals()` so the examples can see the names defined in the notebook, and `verbose=True` to see every example (by default only failures print):

In [ ]:
import doctest

doctest.run_docstring_examples(apply_discount, globals(), verbose=True, name="apply_discount")

A failing example prints the expected and actual output. Here the docstring is wrong (that is the point of doctest — the *docs* are being tested):

In [ ]:
def word_count(text: str) -> int:
    """
    >>> word_count("hello big world")
    3
    >>> word_count("")
    1
    """
    return len(text.split())

doctest.run_docstring_examples(word_count, globals(), name="word_count")

`run_docstring_examples` only prints. When you want a pass/fail **result object**, use the two building blocks that `testmod` is made of: `DocTestFinder` extracts `DocTest` objects (one per docstring), and `DocTestRunner` runs them and keeps a tally:

In [ ]:
finder = doctest.DocTestFinder()
runner = doctest.DocTestRunner(verbose=False)

for func in (apply_discount, word_count):
    for test in finder.find(func, func.__name__, globs=globals()):
        result = runner.run(test, out=lambda s: None)    # silence the failure report
        print(f"{test.name}: {result}")

print(runner.summarize(verbose=False))

## Testing a whole module with `testmod`

In a real project, a module tests its own docstrings with `testmod()` — the classic idiom is at the bottom of the module:

```python
# pricing.py
def double(x): ...

if __name__ == "__main__":
    import doctest
    doctest.testmod()          # tests every docstring in this module
```

```bash
python pricing.py -v           # run via the __main__ block, verbose
python -m doctest -v pricing.py   # no __main__ block needed
python -m doctest README.txt   # text files work too
```

⚠️ **Gotcha:** calling `doctest.testmod()` with no argument in a notebook tests the kernel's `__main__` namespace — the docstrings of everything defined in *any* cell so far, including the deliberately failing `word_count` above. Pass it an explicit module instead. To simulate a real module here, we build one from source with `types.ModuleType`:

In [ ]:
import textwrap
import types

# opening quotes on their own line: see the IPython gotcha below
pricing_source = textwrap.dedent(
    '''
    """Pricing helpers.

    >>> double(21)
    42
    """

    def double(x):
        """Double a number (or repeat a sequence).

        >>> double(4)
        8
        >>> double([1, 2])
        [1, 2, 1, 2]
        """
        return 2 * x
    '''
)

pricing = types.ModuleType("pricing")
exec(pricing_source, pricing.__dict__)

print(doctest.testmod(pricing))                     # TestResults(failed=0, attempted=3)
print(doctest.testmod(pricing, verbose=False).attempted)

`testmod` returns a `TestResults(failed, attempted)` named tuple, so you can assert on it in CI scripts.

⚠️ **Gotcha (notebooks only):** IPython strips `>>> ` and `... ` prompts from cell source before running it, so that you can paste REPL sessions and doctests straight into a cell. Its heuristic leaves the prompts alone inside a triple-quoted string only when the opening quotes start their own line (as in a docstring, or in the cell above). Write `text = textwrap.dedent('''` on one line instead and the prompts silently vanish from the string — doctest then reports `attempted=0`. Always check that `attempted` is what you expect. Plain `.py` files are never affected.

## Output-matching rules

doctest compares **text**, character by character, after running the example. That has consequences:

| Situation | What happens | Fix |
|---|---|---|
| Expression returns a value | its `repr()` is compared (`'ab'` with quotes, `90.0` not `90`) | write exactly what the REPL shows |
| Expression returns `None` | nothing is printed | expect no output |
| Output contains a blank line | a blank line would end the expected block | write `<BLANKLINE>` |
| Object `repr` with a memory address | `<Order object at 0x10a...>` changes every run | `ELLIPSIS` directive, or print something stable |
| Set / unordered output | iteration order may vary | print `sorted(...)` |
| Floats | `0.1 + 0.2` shows `0.30000000000000004` | `round()` or format the value |
| Trailing whitespace | must match exactly | `NORMALIZE_WHITESPACE` |

In [ ]:
def describe(items):
    """
    >>> describe(["b", "a"])
    Items:
    <BLANKLINE>
    a, b
    >>> sorted({"pear", "apple", "fig"})
    ['apple', 'fig', 'pear']
    >>> round(0.1 + 0.2, 2)
    0.3
    >>> print(None)
    None
    >>> None
    """
    print("Items:")
    print()
    print(", ".join(sorted(items)))

doctest.run_docstring_examples(describe, globals(), name="describe")
print("no output above means every example passed")

## Directives: `ELLIPSIS` and `NORMALIZE_WHITESPACE`

Option flags loosen the comparison. Enable them per example with a `# doctest: +FLAG` comment, or for all examples with `optionflags=` on `testmod` / `run_docstring_examples` / `DocTestRunner`.

- `+ELLIPSIS` — `...` in the expected output matches any text (like a `*` wildcard)
- `+NORMALIZE_WHITESPACE` — any run of whitespace (including newlines) matches any other run
- others: `+SKIP` (don't run), `+IGNORE_EXCEPTION_DETAIL` (ignore the message after the exception type)

In [ ]:
class Order:
    def __init__(self, sku):
        self.sku = sku

def make_orders(n):
    """
    >>> make_orders(1)[0]                      # doctest: +ELLIPSIS
    <...Order object at 0x...>
    >>> list(range(30))                        # doctest: +ELLIPSIS
    [0, 1, 2, ..., 29]
    >>> [o.sku for o in make_orders(3)]        # doctest: +NORMALIZE_WHITESPACE
    ['SKU-0',
     'SKU-1',
     'SKU-2']
    >>> import os; os.urandom(4)               # doctest: +SKIP
    b'random every time'
    """
    return [Order(f"SKU-{i}") for i in range(n)]

doctest.run_docstring_examples(make_orders, globals(), name="make_orders")
print("all directive examples passed")

⚠️ **Gotcha:** directives are written *inside the docstring*, so a backslash in an example (e.g. `"a\nb"`) is processed by the docstring itself first. Use a **raw docstring** (`r"""..."""`) for docstrings with backslashes in examples.

## Exceptions in doctests

To document that something raises, write the `Traceback (most recent call last):` header, any indented line(s) (conventionally just `...` — the stack is ignored), and then the **last line** of the traceback: `ExceptionType: message`. Only the header and the last line are compared.

In [ ]:
def parse_quantity(text: str) -> int:
    """
    >>> parse_quantity("3")
    3
    >>> parse_quantity("-1")
    Traceback (most recent call last):
        ...
    ValueError: quantity must be positive, got -1
    >>> parse_quantity("lots")
    Traceback (most recent call last):
        ...
    ValueError: invalid literal for int() with base 10: 'lots'
    >>> parse_quantity("0")                    # doctest: +IGNORE_EXCEPTION_DETAIL
    Traceback (most recent call last):
    ValueError: the message does not matter here
    """
    qty = int(text)
    if qty <= 0:
        raise ValueError(f"quantity must be positive, got {qty}")
    return qty

doctest.run_docstring_examples(parse_quantity, globals(), verbose=False, name="parse_quantity")
print("all exception examples passed")

## Testing documentation files with `testfile`

`doctest.testfile(path)` treats a whole text file (Markdown, reStructuredText, plain text) as one big docstring — perfect for keeping a README or tutorial correct. Everything that is not a `>>>` example or expected output is just prose.

```text
Shopping cart guide
===================

Create a cart and add items:

>>> cart = []
>>> cart.append("apple")
>>> cart
['apple']

The cart length is the number of items:

>>> len(cart)
1
```

By default the path is relative to the *calling module's* directory; pass `module_relative=False` to use a normal path. Here we write the guide into a temporary directory and test it:

In [ ]:
import tempfile
from pathlib import Path

guide = textwrap.dedent(
    """\
    Shopping cart guide
    ===================

    >>> cart = []
    >>> cart.append("apple")
    >>> cart
    ['apple']
    >>> len(cart)
    1
    """
)

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "guide.txt"
    path.write_text(guide, encoding="utf-8")
    print(doctest.testfile(str(path), module_relative=False))

## Doctests inside a `unittest` run

`doctest.DocTestSuite(module)` wraps every docstring of a module as a `unittest` test case, so doctests run alongside your normal tests with `python -m unittest` *(see Standard Library › Testing & Debugging › 001 · Unit Testing with unittest)*. In a test module you hook it in with the `load_tests` protocol:

```python
# tests/test_docs.py
import doctest
import pricing

def load_tests(loader, tests, ignore):
    tests.addTests(doctest.DocTestSuite(pricing))
    return tests
```

In [ ]:
import io
import unittest

suite = doctest.DocTestSuite(pricing)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print(suite.countTestCases(), "doctest cases;", "successful:", result.wasSuccessful())

**Tip:** pytest runs doctests too, with `pytest --doctest-modules` (and `--doctest-glob="*.md"` for text files).

## When to use doctest (and when not)

**Good fit:**

- short usage examples in docstrings of small, pure functions (parsers, formatters, converters)
- keeping README / tutorial snippets correct
- showing the shape of a return value to readers

**Poor fit:**

- anything needing setup/teardown, fixtures, mocks, files, network or time — use `unittest` or pytest
- many edge cases: a docstring full of corner cases is bad documentation *and* a clumsy test suite
- output that is nondeterministic or long (object ids, timestamps, large structures)
- precise numeric checks — `assertAlmostEqual` is clearer than formatted floats

Rule of thumb: **doctests are documentation that happens to be tested**, not your test suite.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Example in API docs | `/// <example><code>` (never run) | `>>>` lines in the docstring |
| Check one function (notebook) | — | `doctest.run_docstring_examples(f, globals(), verbose=True)` |
| Get a pass/fail result | — | `DocTestFinder().find(obj)` + `DocTestRunner().run(test)` |
| Check a module | — | `doctest.testmod(module)` / `python -m doctest -v file.py` |
| Check a text/README file | — | `doctest.testfile(path, module_relative=False)` |
| Wildcard in output | — | `# doctest: +ELLIPSIS` and `...` |
| Ignore whitespace layout | — | `# doctest: +NORMALIZE_WHITESPACE` |
| Expected exception | `Assert.Throws<T>` (in a real test) | `Traceback (most recent call last):` + `...` + `Type: message` |
| Run with the test suite | — | `doctest.DocTestSuite(module)` + `load_tests` |

## Exercises

1. Write `celsius_to_fahrenheit(c: float) -> float` with a docstring containing three doctest examples (`0`, `100`, `-40`), and verify it with `run_docstring_examples` in verbose mode.
2. Write `initials(full_name: str) -> str` (e.g. `"ada lovelace"` → `"A.L."`) with a doctest that documents the `IndexError` raised for an empty string, using the `Traceback` header and the last line only.
3. Write `squares(n)` returning `[i * i for i in range(n)]` with a doctest for `squares(50)` that uses `+ELLIPSIS` so the expected output fits on one short line. Use `DocTestFinder` / `DocTestRunner` to get a `TestResults` and print it.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import doctest

# 1
def celsius_to_fahrenheit(c: float) -> float:
    """
    >>> celsius_to_fahrenheit(0)
    32.0
    >>> celsius_to_fahrenheit(100)
    212.0
    >>> celsius_to_fahrenheit(-40)
    -40.0
    """
    return c * 9 / 5 + 32

doctest.run_docstring_examples(celsius_to_fahrenheit, globals(), verbose=True,
                               name="celsius_to_fahrenheit")

# 2
def initials(full_name: str) -> str:
    """
    >>> initials("ada lovelace")
    'A.L.'
    >>> initials("")
    Traceback (most recent call last):
        ...
    IndexError: string index out of range
    """
    return "".join(part[0].upper() + "." for part in full_name.split(" "))

doctest.run_docstring_examples(initials, globals(), name="initials")

# 3
def squares(n):
    """
    >>> squares(50)          # doctest: +ELLIPSIS
    [0, 1, 4, 9, ..., 2401]
    """
    return [i * i for i in range(n)]

runner = doctest.DocTestRunner()
for test in doctest.DocTestFinder().find(squares, "squares", globs=globals()):
    print(runner.run(test))
```

</details>